In [1]:
# -------------------------------------------------------------------------------
# Указание файла данных, времени и диапазона величин
# -------------------------------------------------------------------------------

dir_in = '../data'
# Файл создается с помощью: 1_create_mat_eqCat_file.ipynb
file_in = 'NEIC_Global_Tohoku_2011.mat'
# completeness magntiude = Mmin, and Mmax (не обязательно указывать)
Mmin, Mmax = 3, None
tmin, tmax = 1900, 2025
# Импорт внешних библиотек
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import os
import scipy.io

# Импорт внутренних модулей
from src.EqCat import EqCat # EqCat — объект Python, который используется для обработки каталогов
from src import clustering
eqCat = EqCat()

# -------------------------------------------------------------------------------
# Загрузка данных с первичной фильтрацией
# -------------------------------------------------------------------------------

eqCat.loadMatBin(f"{dir_in}/{file_in}")
print('Всего событий —', eqCat.size())
eqCat.selectEvents(Mmin, Mmax, 'Mag')
eqCat.selectEvents(tmin, tmax, 'Time')
print('Всего событий после фитрации по магнитуде и времени —', eqCat.size())
if hasattr(eqCat, 'data') and all(key in eqCat.data for key in ['Lon', 'Lat', 'Mag']):
    # Определение границы карты на основе данных
    projection = ccrs.PlateCarree()
    xmin, xmax = eqCat.data['Lon'].min(), eqCat.data['Lon'].max()
    ymin, ymax = eqCat.data['Lat'].min(), eqCat.data['Lat'].max()

    # Добавление отступов для лучшего отображения
    margin_x = (xmax - xmin) * 0.05
    margin_y = (ymax - ymin) * 0.05
    xmin, xmax = xmin - margin_x, xmax + margin_x
    ymin, ymax = ymin - margin_y, ymax + margin_y

    # Настройка фигуры и оси с проекцией PlateCarree
    plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=projection)
    ax.set_extent([xmin, xmax, ymin, ymax], crs=projection)

    # Добавление элементов карты
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray')

    # Отрисовка всех событий как чёрных точек
    ax.plot(eqCat.data['Lon'], eqCat.data['Lat'], 'ko', markersize=0.8, alpha=0.6, label='Все события', transform=projection)

    # Выделение событий с магнитудой >= 6 красными окружностями
    sel6 = eqCat.data['Mag'] >= 6
    ax.plot(eqCat.data['Lon'][sel6], eqCat.data['Lat'][sel6], 'ro', markersize=10, markeredgewidth=1.5, mfc='none', label='Магнитуда ≥ 6', transform=projection)

    # Добавление меридиан и параллелей
    meridians = np.linspace(xmin, xmax, 5)
    parallels = np.linspace(ymin, ymax, 5)
    gl = ax.gridlines(
        xlocs=meridians, ylocs=parallels, 
        draw_labels={'bottom': 'x', 'left': 'y'},  # Метки только снизу (x) и слева (y)
        linewidth=0.5, color='gray', linestyle='--'
    )

    # НаНастройка стиля меток
    gl.xlabel_style = {'size': 10, 'color': 'black'}
    gl.ylabel_style = {'size': 10, 'color': 'black'}

    # Добавление заголовка и легенды
    plt.title('Распределение землетрясений', fontsize=16, pad=20)
    plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)

    # Отображение
    plt.tight_layout()
    plt.show()
else:
    print("Не удалось отобразить график: данные в eqCat.data отсутствуют или неполные.")
# Установка параметров
dPar = {
    'D': 1.6,                 # TODO: необходимы реальные значения, полученные из анализа
    'b': 1.3333398574492394,  # b-значение, вычисляется с помощью специального инструмента
    'Mc': Mmin,               # пороговая магнитуда (минимальная магнитуда, выше которой данные полны)
    'eta_binsize': 0.3,       # размер ячейки гистограммы по логарифмической шкале расстояний
    'xmin': -13,              # минимальное значение по оси X (логарифм расстояния)
    'xmax': 0                 # максимальное значение по оси X
}

# ---------------------------------------------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------------------------------------------

# Добавление глубины как координаты Z
eqCat.data['Z'] = eqCat.data['Depth']
print('Диапазон глубин: от', eqCat.data['Z'].min(), 'до', eqCat.data['Z'].max(), '(км)')

# ---------------------------------------------------------------------------------------------------------
# Вычисление расстояний в пространстве-времени-магнитуде (NND, ближайший сосед)
# ---------------------------------------------------------------------------------------------------------

# Вычисление расстояния до ближайшего соседа (NND) в масштабированном пространстве
dNND = clustering.NND_eta(eqCat, dPar, correct_co_located=True, verbose=True)

# Построение гистограммы логарифмов расстояний до ближайшего соседа
aBins = np.arange(-13, 1, dPar['eta_binsize'], dtype=float)  # создание массива границ ячеек
aHist, aBins = np.histogram(np.log10(dNND['aNND'][dNND['aNND'] > 0]), aBins)  # гистограмма

# Центирование ячейки (по середине интервала)
aBins = aBins[:-1] + dPar['eta_binsize'] * 0.5

# Корекция гистограммы на размер ячейки (чтобы получить плотность)
aHist = aHist / dPar['eta_binsize']

# Нормирование на общее количество событий → выводим оценку плотности вероятности (PDF)
aHist /= eqCat.size()

# ---------------------------------------------------------------------------------------------------------
# Сохранение результатов
# ---------------------------------------------------------------------------------------------------------

# Сохранение данных в формат .mat (MATLAB)
NND_file = f"data/{file_in.split('.')[0]}_NND_Mc_{dPar['Mc']:.1f}.mat"
print('Сохранение файла:', NND_file)

# Сохранение словаря dNND (включая расстояния до соседей) в .mat-файл с сжатием
scipy.io.savemat(NND_file, dNND, do_compression=True)

# ---------------------------------------------------------------------------------------------------------
# Построение гистограммы
# ---------------------------------------------------------------------------------------------------------

# Загрузка значения eta_0 (характерное расстояние, связанное с кластеризацией)
eta_0_file = f"data/{file_in}_Mc_{dPar['Mc']:.1f}_eta_0.txt"

if os.path.isfile(eta_0_file):
    print('Загружаем eta_0 из файла', eta_0_file)
    f_eta_0 = np.loadtxt(eta_0_file, dtype=float)  # читаем значение eta_0
    print('eta_0 =', f_eta_0)
else:
    # Если файл не найден — используем значение по умолчанию
    f_eta_0 = -5
    print('Файл с eta_0 не найден:', eta_0_file, '— используем значение по умолчанию:', f_eta_0)

# Создание графика
fig, ax = plt.subplots()

# Построение гистограммы как столбчатой диаграммы
ax.bar(aBins, aHist,
       width=0.8 * dPar['eta_binsize'],  # ширина столбцов
       align='edge',                     # выравнивание по краю
       color='.5',                       # серый цвет
       label='Mc = %.1f' % dPar['Mc'])   # подпись: пороговая магнитуда

# Вертикальная синяя линия — общее количество событий (N_tot)
ax.plot([f_eta_0, f_eta_0], ax.get_ylim(), 'b-', lw=2,
        label=r'$N_\mathrm{tot}$=%i' % eqCat.size())

# Вертикальная красная пунктирная линия — количество кластерных событий (N_cl)
ax.plot([f_eta_0, f_eta_0], ax.get_ylim(), 'r--', lw=2,
        label=r'$N_\mathrm{cl}$=%i' % dNND['aNND'][dNND['aNND'] < 1e-5].shape[0])

# Размещаем легенду в верхнем левом углу
ax.legend(loc='upper left')

# Подписи осей
ax.set_xlabel(r'NND, $\log_{10} \eta$')  # NND — расстояние до ближайшего соседа
ax.set_ylabel('Количество событий')

# Включаем сетку
ax.grid(True)

# Устанавливаем пределы по оси X
ax.set_xlim(dPar['xmin'], dPar['xmax'])

# Отображаем график
plt.show()

dPar['eta_0'] = f_eta_0
print('Порог сходства', dPar['eta_0'])

clust_file = file_in.replace('all.mat', f'Mc_{dPar["Mc"]:.1f}_clusters.mat')
    
dNND['aNND'] = np.log10(dNND['aNND'])
# Кластеризация по критериям сходства eta_0
dClust = clustering.compileClust( dNND, f_eta_0, useLargerEvents = False)
# ---------------------------------------------------------------------------------------------------------
# Сохранение результатов
# ---------------------------------------------------------------------------------------------------------
scipy.io.savemat(os.path.join( dir_in,clust_file), dClust, do_compression=True)

# Плотность пар событий в r-T
# ------------------------------------------------------------------
catChild = EqCat()
catParent = EqCat()
catChild.copy(eqCat)
catParent.copy(eqCat)
catChild.selEventsFromID(dNND['aEqID_c'], repeats=True)
catParent.selEventsFromID(dNND['aEqID_p'], repeats=True)
print('Размер каталога потомков', catChild.size()),  
print('Размер каталога родителей', catParent.size())  
# Вычисление масштабированных межсобытийных времен и расстояний
a_R, a_T = clustering.rescaled_t_r(catChild, catParent, dPar)
# Построение графика плотности пар событий 
fig = clustering.plot_R_T( a_T, a_R, f_eta_0)

# ----------------------------------------------------------------------------------------------------------------------------------------------------------
# Расширяющееся дерево
# ----------------------------------------------------------------------------------------------------------------------------------------------------------

plt.figure(1)
ax = plt.subplot(111)  
for iEv in range(catParent.size()):
    print(f"MS-ID, {int(catParent.data['N'][iEv]):d}, t-Par: {catParent.data['Time'][iEv]:.5f}, 't-child', {eqCat.data['Time'][iEv]:.5f}", end="\r")

    if dNND['aNND'][iEv] < dPar['eta_0']:  # триггерный кластер
        ax.plot([catParent.data['Time'][iEv]], [catParent.data['Lat'][iEv]], 'ro', ms=12, alpha=.2)
        ax.plot([catParent.data['Time'][iEv], catChild.data['Time'][iEv]],
                 [catParent.data['Lat'][iEv], catChild.data['Lat'][iEv]], 'k-', marker='o', ms=4, mew=1, mfc='none')
    else:  # независимые события
        ax.plot([catChild.data['Time'][iEv]], [catChild.data['Lat'][iEv]], 'bo', ms=5, alpha=.6)
ax.set_xlabel('Время')
ax.set_ylabel('Широта')

# -----------------------------------------------------------------------------------------
# Загрузка данных, выбор событий
# -----------------------------------------------------------------------------------------

N_tot = eqCat.size()
print( 'Общее количество событий', N_tot)

# -----------------------------------------------------------------------------------------
# Одиночные события считаются главными толчками с 0 афтершоками
# -----------------------------------------------------------------------------------------

print('Общее количество кластеров', len( dClust.keys())), 
print('Количество фоновых событий', dClust['0'].shape[0])
a_ID_single  = dClust['0']

# ID фоновых событий
a_iSel       = np.zeros( eqCat.size(), dtype = int)
a_mag_single = np.zeros( len( a_ID_single))
a_N_AS_single= np.zeros( len( a_ID_single))
a_N_FS_single= np.zeros( len( a_ID_single))
for i in range( a_ID_single.shape[0]):
    # ID события может встречаться в каталоге несколько раз
    sel_ev          = eqCat.data['N'] == a_ID_single[i]
    a_mag_single[i] = eqCat.data['Mag'][sel_ev][0]
    a_iSel[sel_ev] = 1 # catalog.data['N'][catalog.data['N']==aEqID[i]][0]
    if sel_ev.sum() != 1:
        error_str = 'Найдено более одного события', eqCat.data['N'][sel_ev]
        raise( ValueError( error_str))
    
# Удаление одиночных событий из каталога
eqCat.selDicAll( np.logical_not(a_iSel))
print('Оставшиеся события', eqCat.size()), 
print('Фоновые события', len(a_mag_single))
dClust.pop('0') # удаление одиночных событий

# ------------------------------------------------------------------------------------------------------------------------
# Получение магнитуд главных толчков с афтершоками, подсчёт афтершоков
# ------------------------------------------------------------------------------------------------------------------------

a_N_FS    = np.zeros(len( dClust.keys()), dtype = int)
a_N_AS    = np.zeros(len( dClust.keys()), dtype = int)
a_MS_mag  = np.zeros(len( dClust.keys()))
a_MS_ID   = np.zeros(len( dClust.keys()), dtype = int)
iCl = 0
for sCl in dClust.keys():
    aEqID = dClust[sCl]
    print('Кластер: ', iCl+1,'из: ', len( dClust.keys()), 'Количество событий в кластере.', 
          len( aEqID), len(np.unique( dClust[sCl])), end="\r")
    
    # Нахождение магнитуды главного толчка и магнитуды всей семьи
    atmp_MAG = np.zeros(len( aEqID))
    atmp_Time= np.zeros(len( aEqID))
    a_iSel   = np.zeros(eqCat.size(), dtype = int)

    # Нахождение для каждой семьи: магнитуды события и времени возникновения
    for iM in range(len( aEqID)):
        sel_ev           = eqCat.data['N'] == aEqID[iM]
        if sel_ev.sum() != 1:
            error_str = 'Найдено более одного или ни одного события', eqCat.data['N'][sel_ev], aEqID[iM]
            raise(ValueError, error_str)
        atmp_MAG[iM]   = eqCat.data['Mag'][sel_ev][0]
        atmp_Time[iM]  = eqCat.data['Time'][sel_ev][0]
        a_iSel[sel_ev] = 1

    # Удаление событий из каталога
    # catalog.selDicAll( np.logical_not(a_iSel))

    # Главный толчок --------------------------------------------------------------- 
    selMS     = atmp_MAG == atmp_MAG.max()
    f_tMS     = atmp_Time[selMS][0]
    i_ID_MS   = aEqID[selMS]
    # print('tMS', tMS, v_currEqID.shape[0], 'MAG', curr_cat.data['MAG'][selMS][0])

    # Афтершоки -------------------------------------------------------------------- 
    selAS     = atmp_Time > f_tMS
    selFS     = atmp_Time < f_tMS
    # print('Количество афтершоков', selAS.sum()
    # Сохранение количества афтершоков для каждой магнитуды главного толчка
    a_MS_mag[iCl] = atmp_MAG[selMS][0]#, dPar['magRound'])
    a_N_AS[iCl]   = selAS.sum()
    a_N_FS[iCl]   = selFS.sum()
    a_MS_ID[iCl]  = int( i_ID_MS[0])
    iCl += 1

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------
# Сравнение главных толчков + одиночных + форшоков + афтершоков с исходным количеством событий в каталоге
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Объединение одиночных событий без афтершоков с главными толчками, у которых есть афтершоки
a_N_FS    = np.append( a_N_FS, a_N_FS_single)
a_N_AS    = np.append( a_N_AS, a_N_AS_single)
a_MS_mag  = np.append( a_MS_mag, a_mag_single)
a_MS_ID   = np.append( a_MS_ID, a_ID_single)
print('Всего событий в каталоге', N_tot,)
print('Всего событий в семьях',a_N_FS.sum() + a_N_AS.sum() + a_MS_mag.shape[0])
#print('Фоновые события', a_mag_single.shape[0], 'Форшоки', a_N_FS_single.sum(), 'Афтершоки', a_N_AS_single.sum(), 'Главные толчки (MS+BG)', a_MS_mag.shape[0]

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------
# Сохранение в текстовый файл ASCII
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------
file_out = '%s/%s_Nas_MS_Mc_%.1f.txt'%(dir_in, file_in.split('.')[0], dPar['Mc'])#, dPar['magRound'])
m_N_as   = np.array([a_MS_mag, a_N_AS, a_N_FS, a_MS_ID])
np.savetxt(f"data/{file_out}", m_N_as.T, fmt='%10.3f%10i%10i%14i',
            header = 'MAG          N-AS          N-FS        MS-ID; note N_AS=0 highlights singles or FS only')

dPar['magRound'] = 1  # биннинг

# Построение графиков --------------------------------------------------------------------==------------------- 
dPar['alpha'] = 1.0  # показатель степени закона
dPar['xmin'] = 2
dPar['xmax'] = 8
dPar['ymin'] = 0.1
dPar['ymax'] = 1e4

# -------------------------------------------------------------------------------------------------------------
# Подсчёт среднего количества афтершоков для магнитуды главного толчка
# -------------------------------------------------------------------------------------------------------------

aMag_round = np.around(m_N_as[0], dPar['magRound'])
aMag_bin = np.array(sorted(np.unique(aMag_round)))
aAveNo_AS = np.ones(len(aMag_bin)) * np.nan
aNo_Fam = np.zeros(len(aMag_bin))  # общее количество семей в бине магнитуды
aNo_AS20 = np.zeros(len(aMag_bin))
aNo_AS80 = np.zeros(len(aMag_bin))

i = 0
for curr_mag in aMag_bin:
    selMag = curr_mag == aMag_round
    aAveNo_AS[i] = m_N_as[1][selMag].mean()
    if selMag.sum() > 0:
        aNo_AS20[i] = np.percentile(m_N_as[1][selMag], 20)
        aNo_AS80[i] = np.percentile(m_N_as[1][selMag], 80)
    aNo_Fam[i] = selMag.sum()
    print(f"Магнитуда: {curr_mag}, Среднее N-АФТЕРШОКОВ: {round(aAveNo_AS[i], 2)}, "
          f"20-й процентиль: {aNo_AS20[i]}, 80-й процентиль: {aNo_AS80[i]}, "
          f"Количество семей: {aNo_Fam[i]}")
    i += 1

# -------------------------------------------------------------------------------------------------------------
# Построение закона продуктивности
# -------------------------------------------------------------------------------------------------------------

plt.figure(1, figsize=(8, 6))
ax = plt.axes([0.14, 0.12, 0.78, 0.83])  # pPlot.createFigureSquare(1)
ax.semilogy(m_N_as[0], m_N_as[1], 'o', ms=6, mew=0, mfc='.7', alpha=0.2)
yerr_low = np.maximum(0, aAveNo_AS - aNo_AS20)
yerr_high = np.maximum(0, aNo_AS80 - aAveNo_AS)
ax.errorbar(aMag_bin, aAveNo_AS, yerr=[yerr_low, yerr_high],
            fmt='o', ecolor='k', elinewidth=0.7, capsize=2.5, mec='k', ms=8, mew=1, mfc='w')

# Экспоненциальная оценка -------------------------------------------------------------------------------------
mag_fit = aMag_bin[min(4, len(aMag_bin) - 1)]  # принудительная подгонка через эту точку
f_no_AS_pl = aAveNo_AS[aMag_bin == mag_fit]
preFac = np.log10(f_no_AS_pl) - dPar['alpha'] * mag_fit
a_N_hat = 10 ** (dPar['alpha'] * aMag_bin + preFac)
ax.semilogy(aMag_bin, a_N_hat, 'w-')
ax.semilogy(aMag_bin, a_N_hat, '-', color='r', lw=2, label='показатель = %.1f' % (np.round(dPar['alpha'], 1)))

# Подписи, пределы и т.д. -------------------------------------------------------------------------------------
ax.set_xlim(dPar['xmin'], dPar['xmax'])
ax.set_ylim(dPar['ymin'], dPar['ymax'])
ax.set_xlabel('Магнитуда главного толчка')
ax.set_ylabel('Количество афтершоков')
ax.legend(loc='upper left', frameon=False);
from matplotlib.colors import LinearSegmentedColormap

# ---------------------------------------------------------------------------------------------
# Визуализация кластеров на географической карте с размерами по магнитуде, цветом по глубине и легендой
# ---------------------------------------------------------------------------------------------

# Синхронизация dNND с текущим каталогом (остаётся без изменений)
if len(dNND['aNND']) != len(eqCat.data['Lon']):
    print(f"Ошибка: Размер dNND['aNND'] ({len(dNND['aNND'])}) не соответствует размеру каталога ({len(eqCat.data['Lon'])}).")
    print("Синхронизируем dNND с текущим каталогом...")
    sel_valid = np.isin(dNND['aEqID_c'], eqCat.data['N'])
    if not np.any(sel_valid):
        print("Ошибка: Нет совпадений между dNND['aEqID_c'] и eqCat.data['N']. Проверьте данные.")
        raise ValueError("Нет совпадений идентификаторов.")
    dNND = {
        'aNND': dNND['aNND'][sel_valid],
        'aEqID_p': dNND['aEqID_p'][sel_valid],
        'aEqID_c': dNND['aEqID_c'][sel_valid],
        'Time': dNND['Time'][sel_valid]
    }

valid_nnd = dNND['aNND'] > 0
dNND['aNND'] = np.maximum(dNND['aNND'], 1e-10)
valid_nnd = dNND['aNND'] > 0

sel_clust = np.zeros_like(dNND['aNND'], dtype=bool)
sel_clust[valid_nnd] = np.log10(dNND['aNND'][valid_nnd]) < dPar['eta_0']

# Синхронизация dClust с текущим каталогом (остаётся без изменений)
for famID in list(dClust.keys()):
    indices = dClust[famID]
    valid_indices = np.isin(indices, eqCat.data['N'])
    dClust[famID] = indices[valid_indices]
    if len(dClust[famID]) == 0:
        print(f"Предупреждение: Кластер {famID} пуст после синхронизации. Удаляем его.")
        del dClust[famID]

num_clusters = len(dClust)

# === Функция преобразования магнитуды в размер маркера (как в другом графике) ===
def mag_to_size(mag):
    if mag < 4:
        return 10
    elif 4 <= mag < 5:
        return 40
    elif 5 <= mag < 6:
        return 100
    elif 6 <= mag < 7:
        return 200
    elif 7 <= mag < 8:
        return 350
    else:
        return 600

# Метки для легенды (как в другом графике)
legend_labels = [
    ('M < 4', 3.5),
    ('4 ≤ M < 5', 4.5),
    ('5 ≤ M < 6', 5.5),
    ('6 ≤ M < 7', 6.5),
    ('7 ≤ M < 8', 7.5),
    ('M ≥ 8', 8.5)
]

# === Цветовая карта по глубине (как в первом графике) ===
max_depth_for_color = 100.0  # км
depth_ticks = [0, 10, 20, 30, 35, 50, 100]
colors = ['#9400D3', '#4B0082', '#0000FF', '#00FF00', '#FFFF00', '#FF7F00', '#FF0000', '#8B0000']

norm_depths = np.array(depth_ticks + [100]) / max_depth_for_color
norm_depths = np.unique(norm_depths[norm_depths <= 1.0])
used_colors = colors[:len(norm_depths)]

cmap_depth = LinearSegmentedColormap.from_list('depth_cmap', list(zip(norm_depths, used_colors)))

# Создание фигуры с географической проекцией
plt.figure(figsize=(12, 10))
ax = plt.axes(projection=ccrs.PlateCarree())

# Добавление географических элементов
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.add_feature(cfeature.COASTLINE, edgecolor='black')
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.STATES, linestyle='--', edgecolor='gray')

# Установление границ карты
lon_min, lon_max = eqCat.data['Lon'].min() - 1, eqCat.data['Lon'].max() + 1
lat_min, lat_max = eqCat.data['Lat'].min() - 1, eqCat.data['Lat'].max() + 1
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# Отображение всех кластеризованных событий с размерами по магнитуде и цветом по глубине
for i, (famID, indices) in enumerate(dClust.items()):
    if len(indices) > 0:
        sel_indices = np.isin(eqCat.data['N'], indices)
        lons = eqCat.data['Lon'][sel_indices]
        lats = eqCat.data['Lat'][sel_indices]
        mags = eqCat.data['Mag'][sel_indices]
        depths = np.abs(eqCat.data['Depth'][sel_indices])  # или 'Depth' — смотрите, как называется столбец
        sizes = [mag_to_size(m) for m in mags]
        
        # Убедитесь, что столбец глубины называется правильно: 'Dep' или 'Depth' или 'depth'
        # Замените 'Dep' на правильное имя столбца, если нужно
        scatter = ax.scatter(
            lons,
            lats,
            s=sizes,
            c=depths,  # ← цвет зависит от глубины
            cmap=cmap_depth,
            alpha=0.8,
            edgecolor='k',
            linewidth=0.3,
            vmin=0,
            vmax=max_depth_for_color,
            transform=ccrs.PlateCarree()
        )

# Сетка с подписями
ax.gridlines(draw_labels=['left', 'bottom'], dms=True, x_inline=False, y_inline=False)

# === Легенда по магнитуде (как в другом графике) ===
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', label=label,
               markerfacecolor='orange',  # цвет точек на легенде (не важен, но пусть будет оранжевый)
               markersize=np.sqrt(mag_to_size(mag)),
               markeredgecolor='k',
               markeredgewidth=0.5)
    for label, mag in legend_labels
]
ax.legend(handles=legend_elements, title='Magnitude', loc='upper right', fontsize=8, frameon=True)

# === Цветовая шкала по глубине ===
cbar = plt.colorbar(scatter, ax=ax, pad=0.03, shrink=0.8) 
cbar.set_label('Earthquake Depth (km)', fontsize=11)
cbar.ax.tick_params(labelsize=10)
cbar.ax.invert_yaxis() 

# Заголовок
ax.set_title('Spatial distribution of clustered earthquakes', fontsize=14)

# Сохранение и отображение
plt.tight_layout()

plt.savefig('plt/cluster_map_Tohoku.jpg', dpi=300, bbox_inches='tight', facecolor='white')

plt.show()

# -----------------------------------------------------------------------------------------
# СРАВНЕНИЕ С ДРУГИМИ МЕТОДАМИ КЛАСТЕРИЗАЦИИ (из make_aftershocks_list_Tohoku.ipynb)
# -----------------------------------------------------------------------------------------

# Параметры главного толчка (Tohoku 2011)
mainshock = {
    'latitude': 38.322,  # Примерные координаты, скорректируйте по необходимости
    'longitude': 142.369,
    'time': '2011-03-11 05:46:24'  # UTC
}

# Предполагаем, что eqCat.data содержит необходимые поля: Lat, Lon, Depth, Mag, Time
# Создаем DataFrame для совместимости с методами из notebook
import pandas as pd
from datetime import datetime
eq_df = pd.DataFrame({
    'latitude': eqCat.data['Lat'],
    'longitude': eqCat.data['Lon'],
    'depth': eqCat.data['Depth'],
    'mag': eqCat.data['Mag'],
    'time': pd.to_datetime([datetime.fromtimestamp(t) for t in eqCat.data['Time']])  # Предполагаем Unix timestamp; скорректируйте если нужно
})

# Фильтрация по времени и расстоянию (2 месяца, 500 км)
from dateutil.relativedelta import relativedelta
from math import radians, sin, cos, atan2, sqrt

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

start_date = pd.to_datetime(mainshock['time']).tz_localize('UTC') - relativedelta(days=1)
end_date = start_date + relativedelta(months=2)

df_time = eq_df[(eq_df['time'] >= start_date) & (eq_df['time'] <= end_date)].copy()
df_time['distance_km'] = df_time.apply(
    lambda row: haversine_distance(mainshock['latitude'], mainshock['longitude'], row['latitude'], row['longitude']),
    axis=1
)
filtered_df = df_time[df_time['distance_km'] <= 500].copy()

# Удаление дубликатов (близкие события)
from sklearn.metrics import pairwise_distances
def remove_close_events(df, dist_thresh=1.0, time_thresh_minutes=10):
    coords = np.radians(df[['latitude', 'longitude']].values)
    dists = pairwise_distances(coords) * 6371  # Haversine approx
    time_diffs = np.abs(np.subtract.outer(df['time'].values, df['time'].values)) / np.timedelta64(1, 'm')
    np.fill_diagonal(dists, np.inf)
    np.fill_diagonal(time_diffs, np.inf)
    close_mask = (dists < dist_thresh) & (time_diffs < time_thresh_minutes)
    to_keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        if to_keep[i]:
            close = np.where(close_mask[i])[0]
            to_keep[close] = False
            to_keep[i] = True
    return df[to_keep].reset_index(drop=True)

cleaned_df = remove_close_events(filtered_df)
print(f"Found {len(cleaned_df)} earthquakes within 2 months and 500 km radius.")

# 1. Метод Molchan-Dmitrieva (текущий метод скрипта) - aftershocks_md
sel_md = sel_clust  # Из предыдущего расчета
aftershocks_md = cleaned_df[sel_md[:len(cleaned_df)]].copy()  # Синхронизируем индексы

# 2. Gardner-Knopoff declustering
# Параметры: tau = 10^(a + b*M), r = 10^(c + d*M), типичные: a=2.1, b=1.0, c=-1.0, d=1.4
def gardner_knopoff_aftershocks(df, mainshock):
    aftershocks = pd.DataFrame()
    for _, row in df.iterrows():
        if row['mag'] >= mainshock.get('mag', 9.0):  # Предполагаем Mw mainshock
            tau_days = 10 ** (2.1 + 1.0 * row['mag'])
            r_km = 10 ** (-1.0 + 1.4 * row['mag'])
            t_start = row['time']
            sel = (df['time'] > t_start) & (df['time'] <= t_start + pd.Timedelta(days=tau_days)) & \
                  (df['distance_km'] <= r_km)
            aftershocks = pd.concat([aftershocks, df[sel]])
    return aftershocks.drop_duplicates().reset_index(drop=True)

aftershocks_gk = gardner_knopoff_aftershocks(cleaned_df, {'mag': 9.1, 'time': start_date, 'latitude': mainshock['latitude'], 'longitude': mainshock['longitude']})

# 3. Reasenberg declustering (упрощенная версия)
# Окно: t_after = c * exp(alpha * (M - Mc)), t_before = t_after / 10, p=1 (mainshock first)
def reasenberg_aftershocks(df, mc=3.0, c=0.05, alpha=2.0, p=1.0):
    df_sorted = df.sort_values('time').reset_index(drop=True)
    is_aftershock = np.zeros(len(df_sorted), dtype=bool)
    for i in range(len(df_sorted)):
        if is_aftershock[i]: continue
        M = df_sorted.iloc[i]['mag']
        tau_after = c * np.exp(alpha * (M - mc))  # days
        tau_before = tau_after / 10
        dist_max = 1.0 * np.exp(1.4 * (M - mc))  # km approx
        t_start = df_sorted.iloc[i]['time'] - pd.Timedelta(days=tau_before)
        t_end = df_sorted.iloc[i]['time'] + pd.Timedelta(days=tau_after)
        sel = (df_sorted['time'] >= t_start) & (df_sorted['time'] <= t_end) & \
              (df_sorted['distance_km'] <= dist_max)
        if p == 1.0:  # Mainshock is the first
            sel_after = sel & (df_sorted['time'] > df_sorted.iloc[i]['time'])
        else:
            sel_after = sel.copy()
        is_aftershock[sel_after] = True
    return df_sorted[~is_aftershock].reset_index(drop=True)  # Background, but for aftershocks invert

aftershocks_reas = cleaned_df[~is_aftershock]  # Invert for aftershocks; adjust logic as needed

# 4. DBSCAN
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
def dbscan_aftershocks(df, eps=0.1, min_samples=3):  # eps in degrees, adjust
    coords = df[['latitude', 'longitude', 'depth']].values
    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(coords_scaled)
    labels = db.labels_
    # Aftershocks: non-noise clusters (exclude -1)
    is_aftershock_db = labels != -1
    return df[is_aftershock_db].reset_index(drop=True)

aftershocks_dbscan = dbscan_aftershocks(cleaned_df)

# 5. AftIdent (предполагаем внешний запуск; здесь загружаем из файла, если доступен)
# aftershocks_ai = pd.read_csv('aftmark_aftershocks.csv')  # Загрузите результаты AftIdent
aftershocks_ai = cleaned_df.copy()  # Placeholder; замените на реальные данные

# Границы для карты (из notebook)
boundaries_df = pd.read_csv('data/All_boundaries', sep='\s+', names=['latitude', 'longitude'], skiprows=lambda x: x==0)  # Загрузите boundaries

# Функция для сравнительного графика (из notebook)
def plot_aftershock_methods_map(full_df, ai_df, gk_df, reas_df, dbscan_df, md_df, mainshock_lat, mainshock_lon):
    fig, axs = plt.subplots(2, 3, figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})
    methods = [
        ("Initial catalog", full_df, axs[0, 0]),
        ("AftIdent", ai_df, axs[0, 1]),
        ("Gardner-Knopoff", gk_df, axs[0, 2]),
        ("Reasenberg", reas_df, axs[1, 0]),
        ("DBSCAN", dbscan_df, axs[1, 1]),
        ("Molchan-Dmitrieva (current)", md_df, axs[1, 2])
    ]
    for title, df, ax in methods:
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.gridlines(draw_labels=True)
        if not df.empty:
            ax.scatter(df['longitude'], df['latitude'], color='orange', s=20, label='Aftershocks', alpha=0.7)
        ax.scatter(mainshock_lon, mainshock_lat, color='red', marker='*', s=100, label='Mainshock', zorder=5)
        ax.plot(boundaries_df['longitude'], boundaries_df['latitude'], color='black', label='Boundary Line', linewidth=0.5)
        ax.set_extent([138, 147, 34, 42], crs=ccrs.PlateCarree())
        ax.set_title(title)
        ax.legend(loc='lower left')
    plt.suptitle("Comparison of Aftershock Identification Methods", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig('plt/comparison_methods_Tohoku.jpg', dpi=300, bbox_inches='tight')
    plt.show()

# Вызов сравнения
plot_aftershock_methods_map(
    full_df=cleaned_df,
    ai_df=aftershocks_ai,
    gk_df=aftershocks_gk,
    reas_df=aftershocks_reas,
    dbscan_df=aftershocks_dbscan,
    md_df=aftershocks_md,
    mainshock_lat=mainshock['latitude'],
    mainshock_lon=mainshock['longitude']
)

# Таблица сравнения (количество идентифицированных афтершоков)
comparison_table = pd.DataFrame({
    'Method': ['Initial', 'AftIdent', 'Gardner-Knopoff', 'Reasenberg', 'DBSCAN', 'Molchan-Dmitrieva'],
    'Num Aftershocks': [len(cleaned_df), len(aftershocks_ai), len(aftershocks_gk), len(aftershocks_reas), len(aftershocks_dbscan), len(aftershocks_md)]
})
print("\nСравнение методов:")
print(comparison_table)
comparison_table.to_csv('data/comparison_methods.csv', index=False)

ModuleNotFoundError: No module named 'src'